In [1]:
import os
import sys
from pathlib import Path

In [2]:
if Path.cwd().name == 'notebooks':
    os.chdir("..")

sys.path.append(Path.cwd())

In [3]:
Path.cwd()

WindowsPath('c:/dev/predictive-maintenance-mlops')

Create a client with no server

In [4]:
from fastapi.testclient import TestClient
from turbofan import serve

client = TestClient(serve.app)

try:
    client.get("/health")
except Exception as e:
    print("Error connecting to the FastAPI server:", type(e).__name__, ":", e)

c:\dev\predictive-maintenance-mlops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Error connecting to the FastAPI server: AttributeError : 'State' object has no attribute 'champion'


Reason for error: app.state.champion is set in lifespan, and lifespan only runs when the client is used as a with block. No with, no startup, no model.

With startup, on a real engine

In [5]:
from turbofan.data import RAW_DIR, load_cmapss
from turbofan.features import FEATURE_SENSORS

test_raw = load_cmapss(RAW_DIR / "test_FD001.txt")
engine = test_raw[test_raw["unit"] == 31]
payload = {"unit": 31, "cycles": engine[["cycle"] + FEATURE_SENSORS].to_dict(orient="records")}

with TestClient(serve.app) as client:
    print(client.get("/health").json())
    r = client.post("/predict", json=payload)
    print(r.status_code, r.json())

2026/10/31 23:08:15 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - jaraco-text (current: uninstalled, required: jaraco-text==4.0.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


{'status': 'ok', 'model_version': 1, 'source': 'registry'}
200 {'unit': 31, 'predicted_rul': 12.4, 'at_cycle': 196, 'model_version': 1}


Bad Request

In [6]:
bad = {"unit": 31, "cycles": [{"cycle": 0}]}

with TestClient(serve.app) as client:
    r = client.post("/predict", json=bad)

print(r.status_code)
for error in r.json()["detail"][:3]:
    print(error["loc"], "->", error["msg"])

422
['body', 'cycles', 0, 'cycle'] -> Input should be greater than or equal to 1
['body', 'cycles', 0, 'sensor_2'] -> Field required
['body', 'cycles', 0, 'sensor_3'] -> Field required


Assert condition: assert <condition> does nothing if the condition is true and raises AssertionError if it’s false.

In [7]:
assert r.status_code == 422
print("passed")

# assert r.status_code == 200, "expected a successful prediction"

passed


In [8]:
import numpy as np
from turbofan.features import get_feature_columns

class FakeSchema:
    def __init__(self, names):
        self.names = names
    def input_names(self):
        return self.names

class FakeMetadata:
    def __init__(self, names):
        self.schema = FakeSchema(names)
    def get_input_schema(self):
        return self.schema

class FakeModel:
    def __init__(self, feature_names, answer=42.0):
        self.metadata = FakeMetadata(feature_names)
        self.answer = answer
        self.received = None
    def predict(self, X):
        self.received = X
        return np.full(len(X), self.answer)

In [9]:
fake = FakeModel(get_feature_columns())

def fake_load_champion():
    return {"model": fake, "version": 99, "window": 5,
            "features": serve.check_fits_code(fake, "fake"), "source": "fake"}

original = serve.load_champion
serve.load_champion = fake_load_champion
try:
    with TestClient(serve.app) as client:
        print(client.get("/health").json())
        r = client.post("/predict", json=payload)
        print(r.json())
    print(fake.received.shape)
finally:
    serve.load_champion = original

{'status': 'ok', 'model_version': 99, 'source': 'fake'}
{'unit': 31, 'predicted_rul': 42.0, 'at_cycle': 196, 'model_version': 99}
(1, 45)
